# Skin Lesion Bias Reduction — Colab training

Runs the EfficientNetV2-B0 baseline classifier on a Colab GPU using the project code in `src/` and a Fitzpatrick17k dataset you've already uploaded.

**Order of operations**
1. Confirm GPU + mount Drive (if used)
2. Point the notebook at your code + data
3. Install dependencies
4. Train the baseline (with class weights, unfreeze schedule, save-best-by-val-loss)
5. Evaluate the best checkpoint and render a markdown bias report
6. *Optional* — kick off the cGAN training (placeholder, run later)

## 1. Verify GPU and Colab environment

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB)

import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

Colab: True
torch: 2.10.0+cu128 cuda available: True
GPU: NVIDIA A100-SXM4-40GB
name, memory.total [MiB], memory.free [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB, 40437 MiB


In [2]:
!nvidia-smi

Fri May  1 19:50:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             46W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Connect your code and data

Two common layouts work:

- **Drive layout** — repo and dataset both sit under `MyDrive`. Mount Drive and point `PROJECT_ROOT` at the repo there. Outputs persist between sessions.
- **Local Colab layout** — clone the repo into `/content/` and put the dataset under `/content/dataset/`. Faster I/O, but everything is wiped when the runtime ends.

Edit `PROJECT_ROOT`, `DATASET_CSV`, and `IMAGE_DIR` in the cell below to match where you uploaded things.

In [3]:
# Mount Drive only if you're using the Drive layout. Skip this cell otherwise.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from pathlib import Path

# === EDIT THESE ===
PROJECT_ROOT = Path("/content/drive/MyDrive/SkinLesionBiasReduction")  # repo root containing src/, dataset/, run_*.sh
# IMAGE_DIR    = PROJECT_ROOT / "dataset/images"
IMAGE_DIR    = PROJECT_ROOT / "dataset/images"

DATASET_CSV = PROJECT_ROOT / "dataset/fitzpatrick17k_c.csv"
# ==================

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGE_DIR:   ", IMAGE_DIR,   "exists:", IMAGE_DIR.exists())
assert PROJECT_ROOT.exists(), f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"
assert IMAGE_DIR.exists(),    f"IMAGE_DIR does not exist:    {IMAGE_DIR}"

%cd $PROJECT_ROOT

PROJECT_ROOT: /content/drive/MyDrive/SkinLesionBiasReduction
IMAGE_DIR:    /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images exists: True
/content/drive/MyDrive/SkinLesionBiasReduction


In [5]:
import subprocess, time  
from pathlib import Path

DRIVE_IMAGE_DIR = IMAGE_DIR
LOCAL_IMAGE_DIR = Path("/content/local_images")                                                                                                                                                
LOCAL_IMAGE_DIR.mkdir(parents=True, exist_ok=True)                                                                                                                                            

def _count_files(d):
    r = subprocess.run(f"ls -1 '{d}' 2>/dev/null | wc -l",
                     shell=True, capture_output=True, text=True)
    return int(r.stdout.strip() or 0)

n_source = _count_files(DRIVE_IMAGE_DIR)
n_local  = _count_files(LOCAL_IMAGE_DIR)
print(f"Drive: {n_source} files | Local: {n_local} files "
      f"(need ~{max(0, n_source - n_local)} more)")

if n_local >= n_source > 0:
    print("Local cache is complete — skipping copy.")
else:
    # Parallel copy: ~64 concurrent cp workers amortize Drive's per-file FUSE overhead.
    # `cp -n` = skip files that already exist at the destination, so this resumes
    # from any partial copy (no rm -rf needed).
    print("Parallel-copying from Drive (64 workers)...")
    t0 = time.time()
    cmd = (
        f"cd '{DRIVE_IMAGE_DIR}' && "
        f"find . -maxdepth 1 -type f -print0 | "
        f"xargs -0 -n 64 -P 64 cp -n -t '{LOCAL_IMAGE_DIR}/'"
    )
    subprocess.run(cmd, shell=True, check=True)
    n_local = _count_files(LOCAL_IMAGE_DIR)
    print(f"Done — {n_local} files in {time.time() - t0:.1f}s "
          f"({n_local / max(1, time.time() - t0):.0f} files/s)")
    assert n_local >= n_source, f"Copy incomplete: expected {n_source}, got {n_local}"

IMAGE_DIR = LOCAL_IMAGE_DIR
print(f"IMAGE_DIR → {IMAGE_DIR}")

Drive: 16518 files | Local: 0 files (need ~16518 more)
Parallel-copying from Drive (64 workers)...
Done — 16518 files in 76.7s (215 files/s)
IMAGE_DIR → /content/local_images


In [6]:
# Alternative: if you don't have the repo on Drive, clone it into /content/ and copy your dataset in.
# Uncomment, replace the URL with your fork, and re-run cell `configure-paths` with PROJECT_ROOT=/content/SkinLesionBiasReduction.
# !git clone git@github.com:hoangnam310/SkinLesionBiasReduction.git

In [7]:
!git fetch origin main && git reset --hard origin/main

From https://github.com/hoangnam310/SkinLesionBiasReduction
 * branch            main       -> FETCH_HEAD
Updating files: 100% (46/46), done.
HEAD is now at 10c21c1 Add train/val/test that use the dataset stratified


## 3. Install dependencies

Colab images already include `torch`, `torchvision`, `numpy`, `pandas`, `Pillow`, `tqdm`, `matplotlib`, and `scipy`. The trainer additionally needs **timm** and **scikit-learn** (sklearn is usually preinstalled, timm usually is not). Tensorboard is optional and is also usually preinstalled.

In [8]:
!pip install --quiet timm 'scikit-learn>=1.3'

## 4. Train the baseline classifier

Full training at **224×224** for 40 epochs. The backbone is frozen for the first 3 epochs (head-only warmup), then unfrozen for fine-tuning at a lower LR.

Key flags being used:
- `--class_weights` — inverse-frequency weighted CrossEntropyLoss; the single biggest fix from the prior run's bias analysis.
- `--freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5` — three warmup epochs on the head, then full fine-tune.
- The trainer also auto-saves the best checkpoint by val_loss across all epochs (no extra flag needed).

> **Tip:** For a quick smoke test, set `EPOCHS = 1` and `IMAGE_SIZE = 64` to verify the pipeline works before committing to the full run.

In [9]:
IMAGE_SIZE   = 224
EPOCHS       = 40
BATCH_SIZE   = 32        # 32 fits comfortably on A100 at 224x224; use 16 on a T4.
LR           = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 4

OUTPUT_DIR = PROJECT_ROOT / "outputs/baseline_efficientnet"
print("Outputs will land under:", OUTPUT_DIR)

Outputs will land under: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet


In [44]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{IMAGE_DIR}" \
    --image_size {IMAGE_SIZE} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --weight_decay {WEIGHT_DECAY} \
    --num_workers {NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{OUTPUT_DIR}" \
    --device cuda

Using device: cuda
Image size: 224x224
Train: 7752 samples (243 batches), Val: 1095 (35), Test: 2211 (70)
Pretrained: True
Freeze backbone: True
Training classifier head only (backbone frozen).
Class weights (benign, malignant, non-neoplastic): [2.5136, 2.2607, 0.463]
Epoch [1/40] train_loss=1.0598, train_acc=0.4521, val_loss=1.0324, val_acc=0.5169  ← best
Epoch [2/40] train_loss=1.0120, train_acc=0.5387, val_loss=0.9973, val_acc=0.5607  ← best
Epoch [3/40] train_loss=0.9825, train_acc=0.5404, val_loss=0.9732, val_acc=0.5963  ← best
Backbone unfrozen at epoch 3; continuing fine-tuning with lr=1e-05
Epoch [4/40] train_loss=0.9565, train_acc=0.5738, val_loss=0.9466, val_acc=0.6000  ← best
Epoch [5/40] train_loss=0.9252, train_acc=0.5893, val_loss=0.9232, val_acc=0.6073  ← best
Epoch [6/40] train_loss=0.8852, train_acc=0.6130, val_loss=0.9023, val_acc=0.6210  ← best
Epoch [7/40] train_loss=0.8639, train_acc=0.6280, val_loss=0.8818, val_acc=0.6365  ← best
Epoch [8/40] train_loss=0.8294, tr

## 5. Evaluate the best checkpoint

`train_baseline_efficientnet.py` restores best-by-val-loss weights before the final eval, so the auto-generated `metrics.json` already uses that snapshot. We additionally re-run `evaluate.py` to write `logs/evaluation_metrics.json` (with bias breakdowns) and then render a markdown report.

In [45]:
runs = sorted(OUTPUT_DIR.glob("*/checkpoint.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert runs, f"No checkpoint.pt under {OUTPUT_DIR}"
LATEST_CKPT = runs[0]
print("Latest checkpoint:", LATEST_CKPT)

!python src/evaluate.py \
    --checkpoint "{LATEST_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{IMAGE_DIR}" \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"


!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

from IPython.display import Markdown, display

run_name = LATEST_CKPT.parent.name
report_path = PROJECT_ROOT / "logs" / f"{run_name}_report.md"
print("Report:", report_path)
display(Markdown(report_path.read_text()))

Latest checkpoint: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260501_205016/checkpoint.pt
Using device: cuda
Loading checkpoint from /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260501_205016/checkpoint.pt
Evaluating on 2211 samples (70 batches)
Evaluation Summary
Samples: 2211
Top-1 Accuracy: 0.6888
Macro AUROC:    0.8272
Macro AUPRC:    0.6648

Per-Fitzpatrick breakdown
----------------------------------------------------------------
fitzpatrick_1 (n=455): acc=0.6593 macroAUROC=0.8335 macroAUPRC=0.6719
             benign: n=   55 AUROC=0.7592 AUPRC=0.3886
          malignant: n=   75 AUROC=0.9066 AUPRC=0.7190
     non-neoplastic: n=  325 AUROC=0.8348 AUPRC=0.9082
fitzpatrick_2 (n=736): acc=0.6970 macroAUROC=0.8284 macroAUPRC=0.6781
             benign: n=  102 AUROC=0.7335 AUPRC=0.4030
          malignant: n=  126 AUROC=0.8954 AUPRC=0.7149
     non-neoplastic: n=  508 AUROC=0.8564 AUPRC=0.9164
fitzpatrick_3 (n=429

# Evaluation Report

| Field | Value |
| --- | --- |
| Checkpoint | `/content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260501_205016/checkpoint.pt` |
| Split | test |
| Image size | 224 |
| CSV path | `/content/drive/MyDrive/SkinLesionBiasReduction/dataset/fitzpatrick17k_c.csv` |
| Image dir | `/content/local_images` |
| Generated at | 2026-05-01T20:52:08.368686 |

## Global metrics

| Metric | Value |
| --- | --- |
| Samples | 2211 |
| Top-1 Accuracy | 0.6888 |
| Macro AUROC | 0.8272 |
| Macro AUPRC | 0.6648 |

## Per-Fitzpatrick subgroup

| Fitzpatrick | n | Accuracy | Macro AUROC | Macro AUPRC |
| --- | --- | --- | --- | --- |
| 1 | 455 | 0.6593 | 0.8335 | 0.6719 |
| 2 | 736 | 0.6970 | 0.8284 | 0.6781 |
| 3 | 429 | 0.6597 | 0.7996 | 0.6369 |
| 4 | 355 | 0.7070 | 0.8402 | 0.6822 |
| 5 | 160 | 0.7063 | 0.8211 | 0.6758 |
| 6 | 76 | 0.8289 | 0.8824 | 0.7395 |

### Per-class AUROC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUROC) | malignant (n / AUROC) | non-neoplastic (n / AUROC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.7592 | 75 / 0.9066 | 325 / 0.8348 |
| 2 | 102 / 0.7335 | 126 / 0.8954 | 508 / 0.8564 |
| 3 | 59 / 0.7289 | 67 / 0.8622 | 303 / 0.8078 |
| 4 | 47 / 0.8312 | 35 / 0.8757 | 273 / 0.8136 |
| 5 | 17 / 0.7688 | 17 / 0.8499 | 126 / 0.8445 |
| 6 | 9 / 0.8093 | 5 / 0.9831 | 62 / 0.8548 |

### Per-class AUPRC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUPRC) | malignant (n / AUPRC) | non-neoplastic (n / AUPRC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.3886 | 75 / 0.7190 | 325 / 0.9082 |
| 2 | 102 / 0.4030 | 126 / 0.7149 | 508 / 0.9164 |
| 3 | 59 / 0.3259 | 67 / 0.6856 | 303 / 0.8992 |
| 4 | 47 / 0.5418 | 35 / 0.5798 | 273 / 0.9248 |
| 5 | 17 / 0.4833 | 17 / 0.5927 | 126 / 0.9515 |
| 6 | 9 / 0.4366 | 5 / 0.8211 | 62 / 0.9609 |

## Classification metrics

| Metric | Value |
| --- | --- |
| Accuracy | 0.6888 |
| Balanced accuracy | 0.6599 |
| Macro F1 | 0.6016 |
| Weighted F1 | 0.7133 |

### Per-class

| Class | Precision | Recall | F1 | Support |
| --- | --- | --- | --- | --- |
| benign | 0.3136 | 0.5675 | 0.4039 | 289 |
| malignant | 0.5388 | 0.7046 | 0.6107 | 325 |
| non-neoplastic | 0.8947 | 0.7076 | 0.7902 | 1597 |

### Confusion matrix

| True \ Pred | benign | malignant | non-neoplastic |
| --- | --- | --- | --- |
| benign | 164 | 42 | 83 |
| malignant | 46 | 229 | 50 |
| non-neoplastic | 313 | 154 | 1130 |


## 6. Segmentation experiments — train classifiers on cv2- and SAM2-segmented images

To isolate the effect of segmentation-aware preprocessing, run two more classifiers with the **same hyperparameters** as section 4 — only `--image_dir` changes:

- **cv2 variant** — ROI selection driven by LAB color variation, no SAM model. Fast.
- **SAM2 variant** — ROI selection constrained to SAM2's foreground mask (segmentation-aware crop).

Both are produced at **224×224** so the trainer's transform doesn't have to upscale at load time (which throws away mid-frequency texture detail).

Workflow:
1. Configure paths and segmentation knobs
2. Build the cv2-segmented dir
3. Install SAM2 + download a checkpoint
4. Build the SAM2-segmented dir
5. (Optional) Backfill SAM2 failures with cv2 so all three runs see the same md5s
6. Common training args
7. Train cv2 → train SAM2
8. Evaluate both

In [15]:
# Local-disk dirs for fast IO during training; mirrored back to Drive at the end.
LOCAL_CV2_DIR     = Path("/content/local_images_cv2_224")
LOCAL_SAM2_DIR    = Path("/content/local_images_sam2_224")
LOCAL_SAM3_DIR    = Path("/content/local_images_sam3_224")
LOCAL_MEDSAM3_DIR = Path("/content/local_images_medsam3_224")
DRIVE_CV2_DIR     = PROJECT_ROOT / "dataset/images_cv2_224"
DRIVE_SAM2_DIR    = PROJECT_ROOT / "dataset/images_sam2_224"
DRIVE_SAM3_DIR    = PROJECT_ROOT / "dataset/images_sam3_224"
DRIVE_MEDSAM3_DIR = PROJECT_ROOT / "dataset/images_medsam3_224"

# Segmentation knobs (apply to all backends)
SEG_SIZE  = 384      # SAM + ROI search resolution (downscaled to OUT_SIZE after)
OUT_SIZE  = 224      # final image side; matches the trainer's --image_size
CROP_FRAC = 0.6
MIN_SKIN  = 0.85

(PROJECT_ROOT / "logs").mkdir(parents=True, exist_ok=True)
for d in (LOCAL_CV2_DIR, LOCAL_SAM2_DIR, LOCAL_SAM3_DIR, LOCAL_MEDSAM3_DIR,
          DRIVE_CV2_DIR, DRIVE_SAM2_DIR, DRIVE_SAM3_DIR, DRIVE_MEDSAM3_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("CV2     →", DRIVE_CV2_DIR)
print("SAM2    →", DRIVE_SAM2_DIR)
print("SAM3    →", DRIVE_SAM3_DIR)
print("MedSAM3 →", DRIVE_MEDSAM3_DIR)
print("Source images at:", LOCAL_IMAGE_DIR, f"({_count_files(LOCAL_IMAGE_DIR)} files)")

CV2     → /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images_cv2_224
SAM2    → /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images_sam2_224
SAM3    → /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images_sam3_224
MedSAM3 → /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images_medsam3_224
Source images at: /content/local_images (16518 files)


### 6.1 cv2 segmentation (no SAM, fast)

Builds `images_cv2_224/` from the local cache. Runs in a few minutes on CPU. Resumable.

In [16]:
# !python src/preprocess_segmentation.py \
#     --backend cv2 \
#     --strategy color \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{LOCAL_IMAGE_DIR}" \
#     --output_dir "{LOCAL_CV2_DIR}" \
#     --seg_size {SEG_SIZE} \
#     --out_size {OUT_SIZE} \
#     --crop_frac {CROP_FRAC} \
#     --min_skin {MIN_SKIN} 2>&1 | tee "{PROJECT_ROOT / 'logs' / 'preprocess_cv2_224.log'}"

# # Mirror to Drive so it survives a Colab disconnect
# !mkdir -p "{DRIVE_CV2_DIR}" && rsync -a "{LOCAL_CV2_DIR}/" "{DRIVE_CV2_DIR}/"
# print("cv2 dir:",
#       _count_files(LOCAL_CV2_DIR), "files local /",
#       _count_files(DRIVE_CV2_DIR), "on Drive")

### 6.2 Install SAM2 and fetch the tiny checkpoint

`sam2` is not preinstalled on Colab. The checkpoint and config name must match — `sam2_hiera_tiny.pt` pairs with `sam2_hiera_t.yaml` (the config string is resolved by name inside the sam2 package).

In [17]:
# SAM2_CKPT_DIR  = PROJECT_ROOT / "checkpoints"
# SAM2_CKPT_PATH = SAM2_CKPT_DIR / "sam2_hiera_tiny.pt"
# SAM2_CKPT_DIR.mkdir(parents=True, exist_ok=True)

# if not SAM2_CKPT_PATH.exists():
#     !curl -L -o "{SAM2_CKPT_PATH}" \
#         https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_tiny.pt
# size_mb = SAM2_CKPT_PATH.stat().st_size / 1e6 if SAM2_CKPT_PATH.exists() else 0
# print(f"SAM2 checkpoint: {SAM2_CKPT_PATH} (exists={SAM2_CKPT_PATH.exists()}, {size_mb:.1f} MB)")

# # Install SAM2 itself (Meta's repo). Skip if already installed in this runtime.
# try:
#     import sam2  # noqa: F401
#     print("sam2 already installed")
# except ImportError:
#     !pip install --quiet "git+https://github.com/facebookresearch/sam2.git"
#     import sam2  # noqa: F401
#     print("sam2 installed")

### 6.3 SAM2 segmentation (GPU)

Run the preprocessor with `--backend sam2`. ~30–45 min on a T4, ~10–15 min on an A100 for 16,519 images at `seg_size=384`. Resumable — files already in `--output_dir` are skipped.

The `tee` pipe captures the per-failure `[fail] <md5>: <exc>` lines, so you can audit which images SAM2 dropped this run.

In [18]:
# !python src/preprocess_segmentation.py \
#     --backend sam2 \
#     --strategy color \
#     --sam2_checkpoint "{SAM2_CKPT_PATH}" \
#     --sam2_config sam2_hiera_t.yaml \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{LOCAL_IMAGE_DIR}" \
#     --output_dir "{LOCAL_SAM2_DIR}" \
#     --seg_size {SEG_SIZE} \
#     --out_size {OUT_SIZE} \
#     --crop_frac {CROP_FRAC} \
#     --min_skin {MIN_SKIN} \
#     --device cuda 2>&1 | tee "{PROJECT_ROOT / 'logs' / 'preprocess_sam2_224.log'}"

# !mkdir -p "{DRIVE_SAM2_DIR}" && rsync -a "{LOCAL_SAM2_DIR}/" "{DRIVE_SAM2_DIR}/"
# print("SAM2 dir:",
#       _count_files(LOCAL_SAM2_DIR), "files local /",
#       _count_files(DRIVE_SAM2_DIR), "on Drive")

### 6.4 (Optional but recommended) Backfill SAM2 failures with cv2

`SkinLesionDataset` silently drops md5s whose `.jpg` is missing in `--image_dir`. If SAM2 fails on N images, the SAM2 trainer sees N fewer rows than the cv2 / raw trainers, and the seed-42 split produces a different cohort — breaking the comparison.

This cell re-runs the preprocessor with `--backend cv2` against the **same** `--output_dir`. Files already produced by SAM2 are skipped (resume-mode default), so this only fills in the gaps. After it finishes, `LOCAL_SAM2_DIR` should match `LOCAL_CV2_DIR`'s file count.

In [19]:
# !python src/preprocess_segmentation.py \
#     --backend cv2 \
#     --strategy color \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{LOCAL_IMAGE_DIR}" \
#     --output_dir "{LOCAL_SAM2_DIR}" \
#     --seg_size {SEG_SIZE} \
#     --out_size {OUT_SIZE} \
#     --crop_frac {CROP_FRAC} \
#     --min_skin {MIN_SKIN} 2>&1 | tee -a "{PROJECT_ROOT / 'logs' / 'preprocess_sam2_224.log'}"

# !rsync -a "{LOCAL_SAM2_DIR}/" "{DRIVE_SAM2_DIR}/"
# print("SAM2 dir after backfill:",
#       _count_files(LOCAL_SAM2_DIR), "files local /",
#       _count_files(DRIVE_SAM2_DIR), "on Drive")

### 6.5 Install SAM3 and run text-prompted segmentation

Vanilla SAM3 with the text prompt `"skin lesion"`. The Python package `sam3` is from `facebookresearch/sam3`; weights are pulled by `build_sam3_image_model()` on first call. Output schema and `--seg_size` / `--out_size` are identical to the SAM2 run, so any downstream trainer code that worked on `images_sam2_224/` works on `images_sam3_224/`.

> If `pip install` from the SAM3 GitHub URL below fails, double-check the repo URL — the package is moving and the URL may have changed.

In [20]:
# !pip uninstall numpy -y

In [21]:
# !pip install "numpy<2,>=1.26"
# try:
#     import sam3  # noqa: F401
#     print("sam3 already installed")
# except ImportError:
#     !pip install --quiet "git+https://github.com/facebookresearch/sam3.git"
#     import sam3  # noqa: F401
#     print("sam3 installed")


In [22]:
# !python src/preprocess_segmentation.py \
#     --backend sam3 \
#     --strategy color \
#     --prompt "skin lesion" \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{LOCAL_IMAGE_DIR}" \
#     --output_dir "{LOCAL_SAM3_DIR}" \
#     --seg_size {SEG_SIZE} \
#     --out_size {OUT_SIZE} \
#     --crop_frac {CROP_FRAC} \
#     --min_skin {MIN_SKIN} \
#     --device cuda 2>&1 | tee "{PROJECT_ROOT / 'logs' / 'preprocess_sam3_224.log'}"

# !mkdir -p "{DRIVE_SAM3_DIR}" && rsync -a "{LOCAL_SAM3_DIR}/" "{DRIVE_SAM3_DIR}/"
# print("SAM3 dir:",
#       _count_files(LOCAL_SAM3_DIR), "files local /",
#       _count_files(DRIVE_SAM3_DIR), "on Drive")

### 6.6 Install MedSAM3 and run LoRA-fine-tuned segmentation

`Joey-S-Liu/MedSAM3` = SAM3 + LoRA, fine-tuned on medical imagery. Two prerequisites the install cell sets up:

1. The repo cloned to `/content/MedSAM3` so `infer_sam.py` (which defines `SAM3LoRAInference`) is on `PYTHONPATH`.
2. The LoRA config (`configs/full_lora_config.yaml`) and trained weights (`outputs/sam3_lora_full/best_lora_weights.pt`) present inside the repo.

The trained weights are **not** committed to the MedSAM3 repo — follow that repo's README to download them, or copy them in from Drive.

In [23]:
# MEDSAM3_DIR = Path("/content/MedSAM3")
# if not MEDSAM3_DIR.exists():
#     !git clone --depth 1 https://github.com/Joey-S-Liu/MedSAM3.git "{MEDSAM3_DIR}"
# else:
#     print(f"MedSAM3 already cloned at {MEDSAM3_DIR}")

# # Optional: install MedSAM3's Python deps if its requirements file is present.
# req = MEDSAM3_DIR / "requirements.txt"
# if req.exists():
#     !pip install --quiet -r "{req}"

# MEDSAM3_CONFIG  = MEDSAM3_DIR / "configs/full_lora_config.yaml"
# MEDSAM3_WEIGHTS = MEDSAM3_DIR / "outputs/sam3_lora_full/best_lora_weights.pt"

# # Stash the LoRA weights on Drive so you don't re-download every session.
# DRIVE_MEDSAM3_WEIGHTS = PROJECT_ROOT / "checkpoints/best_lora_weights.pt"
# if not MEDSAM3_WEIGHTS.exists() and DRIVE_MEDSAM3_WEIGHTS.exists():
#     MEDSAM3_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
#     !cp "{DRIVE_MEDSAM3_WEIGHTS}" "{MEDSAM3_WEIGHTS}"
#     print(f"Restored LoRA weights from Drive → {MEDSAM3_WEIGHTS}")

# print(f"Config:  {MEDSAM3_CONFIG} (exists={MEDSAM3_CONFIG.exists()})")
# print(f"Weights: {MEDSAM3_WEIGHTS} (exists={MEDSAM3_WEIGHTS.exists()})")
# assert MEDSAM3_CONFIG.exists(),  "Missing MedSAM3 LoRA config — see the MedSAM3 repo's README."
# assert MEDSAM3_WEIGHTS.exists(), (
#     "Missing MedSAM3 LoRA weights. Either download per the MedSAM3 README and place at "
#     f"{MEDSAM3_WEIGHTS}, or upload to {DRIVE_MEDSAM3_WEIGHTS} on Drive and rerun this cell."
# )

In [24]:
# !ls /usr/local/lib/python3.12/dist-packages/sam3/assets/

In [25]:
# !cd /usr/local/lib/python3.12/dist-packages && \
#   PYTHONPATH="{MEDSAM3_DIR}:{PROJECT_ROOT}" \
#   python "{PROJECT_ROOT}/src/preprocess_segmentation.py" \
#     --backend medsam3 \
#     --strategy color \
#     --prompt "skin lesion" \
#     --medsam3_config "{MEDSAM3_CONFIG}" \
#     --medsam3_weights "{MEDSAM3_WEIGHTS}" \
#     --medsam3_resolution 1008 \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{LOCAL_IMAGE_DIR}" \
#     --output_dir "{LOCAL_MEDSAM3_DIR}" \
#     --seg_size {SEG_SIZE} \
#     --out_size {OUT_SIZE} \
#     --crop_frac {CROP_FRAC} \
#     --min_skin {MIN_SKIN} \
#     --device cuda 2>&1 | tee "{PROJECT_ROOT / 'logs' / 'preprocess_medsam3_224.log'}"

# !mkdir -p "{DRIVE_MEDSAM3_DIR}" && rsync -a "{LOCAL_MEDSAM3_DIR}/" "{DRIVE_MEDSAM3_DIR}/"
# print("MedSAM3 dir:",
#       _count_files(LOCAL_MEDSAM3_DIR), "files local /",
#       _count_files(DRIVE_MEDSAM3_DIR), "on Drive")

In [32]:
# Copy segmented images from Drive → local disk for fast IO
import subprocess, time

copies = [
    ("cv2",     DRIVE_CV2_DIR,     LOCAL_CV2_DIR),
    ("SAM2",    DRIVE_SAM2_DIR,    LOCAL_SAM2_DIR),
    ("MedSAM3", DRIVE_MEDSAM3_DIR, LOCAL_MEDSAM3_DIR),
]

for label, src, dst in copies:
    dst.mkdir(parents=True, exist_ok=True)
    n_src = _count_files(src)
    n_dst = _count_files(dst)
    if n_src == 0:
        print(f"[{label}] ⚠️  Drive dir empty ({src}), skipping.")
        continue
    if n_dst >= n_src:
        print(f"[{label}] ✓ Local cache complete ({n_dst} files).")
        continue
    print(f"[{label}] Copying {n_src} files from Drive → {dst} ...")
    t0 = time.time()
    cmd = (
        f"cd '{src}' && "
        f"find . -maxdepth 1 -type f -print0 | "
        f"xargs -0 -n 64 -P 64 cp -n -t '{dst}/'"
    )
    subprocess.run(cmd, shell=True, check=True)
    n_dst = _count_files(dst)
    print(f"[{label}] Done — {n_dst} files in {time.time() - t0:.1f}s "
          f"({n_dst / max(1, time.time() - t0):.0f} files/s)")


[cv2] Copying 11059 files from Drive → /content/local_images_cv2_224 ...
[cv2] Done — 11058 files in 41.7s (265 files/s)
[SAM2] Copying 11059 files from Drive → /content/local_images_sam2_224 ...
[SAM2] Done — 11058 files in 40.2s (275 files/s)
[MedSAM3] Copying 11059 files from Drive → /content/local_images_medsam3_224 ...
[MedSAM3] Done — 11058 files in 157.2s (70 files/s)


### 6.7 Common training args

These match the section 4 baseline run (`20260427_055032`) **exactly**. The only thing that changes between the three runs is `--image_dir`.

In [33]:
# Mirror section 4 — keep these in sync if you change section 4.
SEG_IMAGE_SIZE   = 224
SEG_EPOCHS       = 40
SEG_BATCH_SIZE   = 32
SEG_LR           = 1e-4
SEG_WEIGHT_DECAY = 1e-4
SEG_NUM_WORKERS  = 4
SEG_OUTPUT_DIR   = OUTPUT_DIR     # all three runs share this parent; each gets its own timestamped subdir
print("Outputs land under:", SEG_OUTPUT_DIR)

Outputs land under: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet


### 6.8 Train classifier on cv2-segmented images

Identical hyperparameters to the section 4 baseline (`20260427_055032`); only `--image_dir` changes.

In [34]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_CV2_DIR}" \
    --image_size {SEG_IMAGE_SIZE} \
    --epochs {SEG_EPOCHS} \
    --batch_size {SEG_BATCH_SIZE} \
    --lr {SEG_LR} \
    --weight_decay {SEG_WEIGHT_DECAY} \
    --num_workers {SEG_NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{SEG_OUTPUT_DIR}" \
    --device cuda

Using device: cuda
Image size: 224x224
Train: 7752 samples (243 batches), Val: 1095 (35), Test: 2211 (70)
Pretrained: True
Freeze backbone: True
model.safetensors: 100% 28.8M/28.8M [00:01<00:00, 25.1MB/s]
Training classifier head only (backbone frozen).
Class weights (benign, malignant, non-neoplastic): [2.5136, 2.2607, 0.463]
Epoch [1/40] train_loss=1.0703, train_acc=0.3989, val_loss=1.0463, val_acc=0.3936  ← best
Epoch [2/40] train_loss=1.0266, train_acc=0.5059, val_loss=1.0159, val_acc=0.4795  ← best
Epoch [3/40] train_loss=1.0014, train_acc=0.5104, val_loss=1.0002, val_acc=0.5059  ← best
Backbone unfrozen at epoch 3; continuing fine-tuning with lr=1e-05
Epoch [4/40] train_loss=0.9857, train_acc=0.5392, val_loss=0.9814, val_acc=0.5352  ← best
Epoch [5/40] train_loss=0.9578, train_acc=0.5512, val_loss=0.9672, val_acc=0.5324  ← best
Epoch [6/40] train_loss=0.9312, train_acc=0.5725, val_loss=0.9516, val_acc=0.5470  ← best
Epoch [7/40] train_loss=0.9162, train_acc=0.5802, val_loss=0.938

### 6.9 Train classifier on SAM2-segmented images

Same args as above, just `--image_dir` points at `LOCAL_SAM2_DIR`.

In [35]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_SAM2_DIR}" \
    --image_size {SEG_IMAGE_SIZE} \
    --epochs {SEG_EPOCHS} \
    --batch_size {SEG_BATCH_SIZE} \
    --lr {SEG_LR} \
    --weight_decay {SEG_WEIGHT_DECAY} \
    --num_workers {SEG_NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{SEG_OUTPUT_DIR}" \
    --device cuda

Using device: cuda
Image size: 224x224
Train: 7752 samples (243 batches), Val: 1095 (35), Test: 2211 (70)
Pretrained: True
Freeze backbone: True
Training classifier head only (backbone frozen).
Class weights (benign, malignant, non-neoplastic): [2.5136, 2.2607, 0.463]
Epoch [1/40] train_loss=1.0701, train_acc=0.3949, val_loss=1.0513, val_acc=0.3781  ← best
Epoch [2/40] train_loss=1.0266, train_acc=0.4997, val_loss=1.0204, val_acc=0.4767  ← best
Epoch [3/40] train_loss=1.0036, train_acc=0.5064, val_loss=1.0055, val_acc=0.5014  ← best
Backbone unfrozen at epoch 3; continuing fine-tuning with lr=1e-05
Epoch [4/40] train_loss=0.9855, train_acc=0.5325, val_loss=0.9873, val_acc=0.5233  ← best
Epoch [5/40] train_loss=0.9585, train_acc=0.5476, val_loss=0.9722, val_acc=0.5260  ← best
Epoch [6/40] train_loss=0.9341, train_acc=0.5744, val_loss=0.9579, val_acc=0.5342  ← best
Epoch [7/40] train_loss=0.9138, train_acc=0.5841, val_loss=0.9440, val_acc=0.5607  ← best
Epoch [8/40] train_loss=0.8926, tr

### 6.10 Train classifier on SAM3-segmented images

Same args as the SAM2 run, just `--image_dir` points at `LOCAL_SAM3_DIR`.

In [36]:
# !python src/train_baseline_efficientnet.py \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{LOCAL_SAM3_DIR}" \
#     --image_size {SEG_IMAGE_SIZE} \
#     --epochs {SEG_EPOCHS} \
#     --batch_size {SEG_BATCH_SIZE} \
#     --lr {SEG_LR} \
#     --weight_decay {SEG_WEIGHT_DECAY} \
#     --num_workers {SEG_NUM_WORKERS} \
#     --class_weights \
#     --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
#     --output_dir "{SEG_OUTPUT_DIR}" \
#     --device cuda

### 6.11 Train classifier on MedSAM3-segmented images

Same args as the SAM2 run, just `--image_dir` points at `LOCAL_MEDSAM3_DIR`.

In [37]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_MEDSAM3_DIR}" \
    --image_size {SEG_IMAGE_SIZE} \
    --epochs {SEG_EPOCHS} \
    --batch_size {SEG_BATCH_SIZE} \
    --lr {SEG_LR} \
    --weight_decay {SEG_WEIGHT_DECAY} \
    --num_workers {SEG_NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{SEG_OUTPUT_DIR}" \
    --device cuda

Using device: cuda
Image size: 224x224
Train: 7752 samples (243 batches), Val: 1095 (35), Test: 2211 (70)
Pretrained: True
Freeze backbone: True
Training classifier head only (backbone frozen).
Class weights (benign, malignant, non-neoplastic): [2.5136, 2.2607, 0.463]
Epoch [1/40] train_loss=1.0708, train_acc=0.3906, val_loss=1.0484, val_acc=0.3653  ← best
Epoch [2/40] train_loss=1.0261, train_acc=0.4977, val_loss=1.0187, val_acc=0.4594  ← best
Epoch [3/40] train_loss=1.0025, train_acc=0.5054, val_loss=1.0009, val_acc=0.5014  ← best
Backbone unfrozen at epoch 3; continuing fine-tuning with lr=1e-05
Epoch [4/40] train_loss=0.9852, train_acc=0.5353, val_loss=0.9807, val_acc=0.5251  ← best
Epoch [5/40] train_loss=0.9586, train_acc=0.5415, val_loss=0.9693, val_acc=0.5233  ← best
Epoch [6/40] train_loss=0.9338, train_acc=0.5734, val_loss=0.9537, val_acc=0.5333  ← best
Epoch [7/40] train_loss=0.9147, train_acc=0.5786, val_loss=0.9391, val_acc=0.5607  ← best
Epoch [8/40] train_loss=0.8921, tr

### 6.12 Evaluate all four segmented runs

Each cell below picks the most recent run whose `args.image_dir` matches the segmented dir, then writes `logs/<run_name>_report.md` and renders it inline. Run them in order — `evaluate.py` overwrites `logs/evaluation_metrics.json` each call, but the per-run markdown report is keyed by run name so all four are kept.

In [40]:
import torch as _torch_for_seg
from IPython.display import Markdown, display

def _seg_latest_under(image_dir: Path, label: str) -> Path:
    """Pick the most recent run under SEG_OUTPUT_DIR whose args.image_dir == image_dir."""
    runs = sorted(SEG_OUTPUT_DIR.glob("*/checkpoint.pt"),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    for ckpt in runs:
        try:
            args = _torch_for_seg.load(ckpt, map_location="cpu", weights_only=False).get("args", {})
        except Exception:
            continue
        if Path(args.get("image_dir", "")) == image_dir:
            print(f"[{label}] {ckpt}")
            return ckpt
    raise FileNotFoundError(f"No checkpoint found for image_dir={image_dir}")

CV2_CKPT = _seg_latest_under(LOCAL_CV2_DIR, "cv2")

!python src/evaluate.py \
    --checkpoint "{CV2_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_CV2_DIR}" \
    --batch_size {SEG_BATCH_SIZE} \
    --num_workers {SEG_NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

cv2_report = PROJECT_ROOT / "logs" / f"{CV2_CKPT.parent.name}_report.md"
print("cv2 report:", cv2_report)
display(Markdown(cv2_report.read_text()))

[cv2] /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260501_201239/checkpoint.pt
Using device: cuda
Loading checkpoint from /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260501_201239/checkpoint.pt
Evaluating on 2211 samples (70 batches)
Evaluation Summary
Samples: 2211
Top-1 Accuracy: 0.6436
Macro AUROC:    0.7818
Macro AUPRC:    0.6033

Per-Fitzpatrick breakdown
----------------------------------------------------------------
fitzpatrick_1 (n=455): acc=0.6242 macroAUROC=0.7848 macroAUPRC=0.6074
             benign: n=   55 AUROC=0.7025 AUPRC=0.2894
          malignant: n=   75 AUROC=0.8569 AUPRC=0.6568
     non-neoplastic: n=  325 AUROC=0.7951 AUPRC=0.8760
fitzpatrick_2 (n=736): acc=0.6291 macroAUROC=0.7829 macroAUPRC=0.6085
             benign: n=  102 AUROC=0.7069 AUPRC=0.2836
          malignant: n=  126 AUROC=0.8397 AUPRC=0.6533
     non-neoplastic: n=  508 AUROC=0.8019 AUPRC=0.8887
fitzpatrick_3 (n=429): acc=0.6434

# Evaluation Report

| Field | Value |
| --- | --- |
| Checkpoint | `/content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260501_201239/checkpoint.pt` |
| Split | test |
| Image size | 224 |
| CSV path | `/content/drive/MyDrive/SkinLesionBiasReduction/dataset/fitzpatrick17k_c.csv` |
| Image dir | `/content/local_images_cv2_224` |
| Generated at | 2026-05-01T20:32:42.304094 |

## Global metrics

| Metric | Value |
| --- | --- |
| Samples | 2211 |
| Top-1 Accuracy | 0.6436 |
| Macro AUROC | 0.7818 |
| Macro AUPRC | 0.6033 |

## Per-Fitzpatrick subgroup

| Fitzpatrick | n | Accuracy | Macro AUROC | Macro AUPRC |
| --- | --- | --- | --- | --- |
| 1 | 455 | 0.6242 | 0.7848 | 0.6074 |
| 2 | 736 | 0.6291 | 0.7829 | 0.6085 |
| 3 | 429 | 0.6434 | 0.7783 | 0.6101 |
| 4 | 355 | 0.6845 | 0.7979 | 0.6219 |
| 5 | 160 | 0.6438 | 0.7429 | 0.5815 |
| 6 | 76 | 0.7105 | 0.7601 | 0.6128 |

### Per-class AUROC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUROC) | malignant (n / AUROC) | non-neoplastic (n / AUROC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.7025 | 75 / 0.8569 | 325 / 0.7951 |
| 2 | 102 / 0.7069 | 126 / 0.8397 | 508 / 0.8019 |
| 3 | 59 / 0.6998 | 67 / 0.8636 | 303 / 0.7716 |
| 4 | 47 / 0.7510 | 35 / 0.8632 | 273 / 0.7794 |
| 5 | 17 / 0.6791 | 17 / 0.8013 | 126 / 0.7481 |
| 6 | 9 / 0.5539 | 5 / 0.9718 | 62 / 0.7546 |

### Per-class AUPRC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUPRC) | malignant (n / AUPRC) | non-neoplastic (n / AUPRC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.2894 | 75 / 0.6568 | 325 / 0.8760 |
| 2 | 102 / 0.2836 | 126 / 0.6533 | 508 / 0.8887 |
| 3 | 59 / 0.2917 | 67 / 0.6622 | 303 / 0.8764 |
| 4 | 47 / 0.4083 | 35 / 0.5507 | 273 / 0.9068 |
| 5 | 17 / 0.3444 | 17 / 0.5110 | 126 / 0.8889 |
| 6 | 9 / 0.2466 | 5 / 0.6593 | 62 / 0.9326 |

## Classification metrics

| Metric | Value |
| --- | --- |
| Accuracy | 0.6436 |
| Balanced accuracy | 0.5956 |
| Macro F1 | 0.5431 |
| Weighted F1 | 0.6701 |

### Per-class

| Class | Precision | Recall | F1 | Support |
| --- | --- | --- | --- | --- |
| benign | 0.2588 | 0.4325 | 0.3238 | 289 |
| malignant | 0.4585 | 0.6800 | 0.5477 | 325 |
| non-neoplastic | 0.8644 | 0.6744 | 0.7577 | 1597 |

### Confusion matrix

| True \ Pred | benign | malignant | non-neoplastic |
| --- | --- | --- | --- |
| benign | 125 | 53 | 111 |
| malignant | 46 | 221 | 58 |
| non-neoplastic | 312 | 208 | 1077 |


In [41]:
SAM2_CKPT = _seg_latest_under(LOCAL_SAM2_DIR, "sam2")

!python src/evaluate.py \
    --checkpoint "{SAM2_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_SAM2_DIR}" \
    --batch_size {SEG_BATCH_SIZE} \
    --num_workers {SEG_NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

sam2_report = PROJECT_ROOT / "logs" / f"{SAM2_CKPT.parent.name}_report.md"
print("SAM2 report:", sam2_report)
display(Markdown(sam2_report.read_text()))

[sam2] /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260501_202117/checkpoint.pt
Using device: cuda
Loading checkpoint from /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260501_202117/checkpoint.pt
Evaluating on 2211 samples (70 batches)
Evaluation Summary
Samples: 2211
Top-1 Accuracy: 0.6572
Macro AUROC:    0.7836
Macro AUPRC:    0.6061

Per-Fitzpatrick breakdown
----------------------------------------------------------------
fitzpatrick_1 (n=455): acc=0.6220 macroAUROC=0.7935 macroAUPRC=0.6115
             benign: n=   55 AUROC=0.7094 AUPRC=0.3020
          malignant: n=   75 AUROC=0.8768 AUPRC=0.6534
     non-neoplastic: n=  325 AUROC=0.7942 AUPRC=0.8791
fitzpatrick_2 (n=736): acc=0.6413 macroAUROC=0.7727 macroAUPRC=0.6054
             benign: n=  102 AUROC=0.6896 AUPRC=0.3001
          malignant: n=  126 AUROC=0.8379 AUPRC=0.6434
     non-neoplastic: n=  508 AUROC=0.7906 AUPRC=0.8727
fitzpatrick_3 (n=429): acc=0.685

# Evaluation Report

| Field | Value |
| --- | --- |
| Checkpoint | `/content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260501_202117/checkpoint.pt` |
| Split | test |
| Image size | 224 |
| CSV path | `/content/drive/MyDrive/SkinLesionBiasReduction/dataset/fitzpatrick17k_c.csv` |
| Image dir | `/content/local_images_sam2_224` |
| Generated at | 2026-05-01T20:33:04.529274 |

## Global metrics

| Metric | Value |
| --- | --- |
| Samples | 2211 |
| Top-1 Accuracy | 0.6572 |
| Macro AUROC | 0.7836 |
| Macro AUPRC | 0.6061 |

## Per-Fitzpatrick subgroup

| Fitzpatrick | n | Accuracy | Macro AUROC | Macro AUPRC |
| --- | --- | --- | --- | --- |
| 1 | 455 | 0.6220 | 0.7935 | 0.6115 |
| 2 | 736 | 0.6413 | 0.7727 | 0.6054 |
| 3 | 429 | 0.6853 | 0.8017 | 0.6349 |
| 4 | 355 | 0.7042 | 0.7848 | 0.6030 |
| 5 | 160 | 0.6312 | 0.7573 | 0.5918 |
| 6 | 76 | 0.6974 | 0.7480 | 0.6058 |

### Per-class AUROC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUROC) | malignant (n / AUROC) | non-neoplastic (n / AUROC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.7094 | 75 / 0.8768 | 325 / 0.7942 |
| 2 | 102 / 0.6896 | 126 / 0.8379 | 508 / 0.7906 |
| 3 | 59 / 0.7429 | 67 / 0.8681 | 303 / 0.7942 |
| 4 | 47 / 0.7347 | 35 / 0.8537 | 273 / 0.7661 |
| 5 | 17 / 0.6993 | 17 / 0.8100 | 126 / 0.7626 |
| 6 | 9 / 0.5887 | 5 / 0.9352 | 62 / 0.7200 |

### Per-class AUPRC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUPRC) | malignant (n / AUPRC) | non-neoplastic (n / AUPRC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.3020 | 75 / 0.6534 | 325 / 0.8791 |
| 2 | 102 / 0.3001 | 126 / 0.6434 | 508 / 0.8727 |
| 3 | 59 / 0.3271 | 67 / 0.6821 | 303 / 0.8955 |
| 4 | 47 / 0.3934 | 35 / 0.5176 | 273 / 0.8982 |
| 5 | 17 / 0.3714 | 17 / 0.5077 | 126 / 0.8963 |
| 6 | 9 / 0.2060 | 5 / 0.6833 | 62 / 0.9281 |

## Classification metrics

| Metric | Value |
| --- | --- |
| Accuracy | 0.6572 |
| Balanced accuracy | 0.6112 |
| Macro F1 | 0.5606 |
| Weighted F1 | 0.6827 |

### Per-class

| Class | Precision | Recall | F1 | Support |
| --- | --- | --- | --- | --- |
| benign | 0.2685 | 0.4637 | 0.3401 | 289 |
| malignant | 0.4966 | 0.6831 | 0.5751 | 325 |
| non-neoplastic | 0.8672 | 0.6869 | 0.7666 | 1597 |

### Confusion matrix

| True \ Pred | benign | malignant | non-neoplastic |
| --- | --- | --- | --- |
| benign | 134 | 45 | 110 |
| malignant | 45 | 222 | 58 |
| non-neoplastic | 320 | 180 | 1097 |


In [43]:
MEDSAM3_CKPT = _seg_latest_under(LOCAL_MEDSAM3_DIR, "medsam3")

!python src/evaluate.py \
    --checkpoint "{MEDSAM3_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_MEDSAM3_DIR}" \
    --batch_size {SEG_BATCH_SIZE} \
    --num_workers {SEG_NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

medsam3_report = PROJECT_ROOT / "logs" / f"{MEDSAM3_CKPT.parent.name}_report.md"
print("medsam3 report:", medsam3_report)
display(Markdown(medsam3_report.read_text()))

[medsam3] /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260501_202948/checkpoint.pt
Using device: cuda
Loading checkpoint from /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260501_202948/checkpoint.pt
Evaluating on 2211 samples (70 batches)
Evaluation Summary
Samples: 2211
Top-1 Accuracy: 0.6364
Macro AUROC:    0.7752
Macro AUPRC:    0.5951

Per-Fitzpatrick breakdown
----------------------------------------------------------------
fitzpatrick_1 (n=455): acc=0.6066 macroAUROC=0.7894 macroAUPRC=0.6074
             benign: n=   55 AUROC=0.7046 AUPRC=0.2934
          malignant: n=   75 AUROC=0.8666 AUPRC=0.6571
     non-neoplastic: n=  325 AUROC=0.7972 AUPRC=0.8716
fitzpatrick_2 (n=736): acc=0.6332 macroAUROC=0.7782 macroAUPRC=0.5964
             benign: n=  102 AUROC=0.7004 AUPRC=0.2775
          malignant: n=  126 AUROC=0.8439 AUPRC=0.6285
     non-neoplastic: n=  508 AUROC=0.7903 AUPRC=0.8833
fitzpatrick_3 (n=429): acc=0.

# Evaluation Report

| Field | Value |
| --- | --- |
| Checkpoint | `/content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260501_202948/checkpoint.pt` |
| Split | test |
| Image size | 224 |
| CSV path | `/content/drive/MyDrive/SkinLesionBiasReduction/dataset/fitzpatrick17k_c.csv` |
| Image dir | `/content/local_images_medsam3_224` |
| Generated at | 2026-05-01T20:33:32.985411 |

## Global metrics

| Metric | Value |
| --- | --- |
| Samples | 2211 |
| Top-1 Accuracy | 0.6364 |
| Macro AUROC | 0.7752 |
| Macro AUPRC | 0.5951 |

## Per-Fitzpatrick subgroup

| Fitzpatrick | n | Accuracy | Macro AUROC | Macro AUPRC |
| --- | --- | --- | --- | --- |
| 1 | 455 | 0.6066 | 0.7894 | 0.6074 |
| 2 | 736 | 0.6332 | 0.7782 | 0.5964 |
| 3 | 429 | 0.6387 | 0.7714 | 0.6102 |
| 4 | 355 | 0.6873 | 0.7770 | 0.5910 |
| 5 | 160 | 0.6062 | 0.7204 | 0.6029 |
| 6 | 76 | 0.6579 | 0.7504 | 0.5884 |

### Per-class AUROC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUROC) | malignant (n / AUROC) | non-neoplastic (n / AUROC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.7046 | 75 / 0.8666 | 325 / 0.7972 |
| 2 | 102 / 0.7004 | 126 / 0.8439 | 508 / 0.7903 |
| 3 | 59 / 0.6689 | 67 / 0.8761 | 303 / 0.7693 |
| 4 | 47 / 0.7184 | 35 / 0.8473 | 273 / 0.7654 |
| 5 | 17 / 0.6405 | 17 / 0.7849 | 126 / 0.7360 |
| 6 | 9 / 0.5721 | 5 / 0.9577 | 62 / 0.7212 |

### Per-class AUPRC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUPRC) | malignant (n / AUPRC) | non-neoplastic (n / AUPRC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.2934 | 75 / 0.6571 | 325 / 0.8716 |
| 2 | 102 / 0.2775 | 126 / 0.6285 | 508 / 0.8833 |
| 3 | 59 / 0.2818 | 67 / 0.6795 | 303 / 0.8694 |
| 4 | 47 / 0.3618 | 35 / 0.5199 | 273 / 0.8912 |
| 5 | 17 / 0.3318 | 17 / 0.5901 | 126 / 0.8867 |
| 6 | 9 / 0.1756 | 5 / 0.6655 | 62 / 0.9240 |

## Classification metrics

| Metric | Value |
| --- | --- |
| Accuracy | 0.6364 |
| Balanced accuracy | 0.5944 |
| Macro F1 | 0.5402 |
| Weighted F1 | 0.6649 |

### Per-class

| Class | Precision | Recall | F1 | Support |
| --- | --- | --- | --- | --- |
| benign | 0.2515 | 0.4464 | 0.3217 | 289 |
| malignant | 0.4620 | 0.6738 | 0.5482 | 325 |
| non-neoplastic | 0.8652 | 0.6631 | 0.7508 | 1597 |

### Confusion matrix

| True \ Pred | benign | malignant | non-neoplastic |
| --- | --- | --- | --- |
| benign | 129 | 54 | 106 |
| malignant | 47 | 219 | 59 |
| non-neoplastic | 337 | 201 | 1059 |


## 7. Quick smoke test (optional)

If you want to verify the pipeline before committing to a full 40-epoch run, temporarily override section **4** with:

```python
IMAGE_SIZE = 64
EPOCHS     = 1
BATCH_SIZE = 128
```

Then re-run sections **4** and **5**. Once it completes without errors, restore the defaults (224 / 40 / 32) and launch the real training.

## 8. Optional — train the cGAN later

The generative side lives in `src/train.py` (vanilla cGAN; the WGAN-GP critic exists in `src/cgan.py` but the WGAN trainer hasn't been committed yet). To run on Colab once you're ready:

In [ ]:
##Outputs land under outputs/<timestamp>/.
# !python src/train.py \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{IMAGE_DIR}" \
#     --epochs 200 \
#     --batch_size 64 \
#     --lr 0.0002 \
#     --output_dir "{PROJECT_ROOT / 'outputs'}" \
#     --device cuda

In [ ]:
# After (or during) cGAN training, watch losses + sample grids in TensorBoard:
# %load_ext tensorboard
# %tensorboard --logdir $PROJECT_ROOT/outputs